<a href="https://colab.research.google.com/github/Sultoniromadhon/data-science-2026/blob/main/Pertemuan10_Sultoni_Romadhon_250401020198.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

* Nama : SULTONI ROMADHON
* NIM : 250401020198
* Kelas : IF403
* Prodi : Informatika PJJ S1

In [2]:
import pandas as pd

# Membaca dataset
df = pd.read_csv("sample_data/Telco-Customer-Churn.csv")

# Melihat ukuran data
print("Ukuran Dataset:", df.shape)

# Informasi dataset
print(df.info())

# Lima data pertama
print(df.head())

# Proporsi kelas target
print("\nProporsi Churn:")
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True))

Ukuran Dataset: (7043, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBillin

In [3]:
# Mengubah TotalCharges menjadi numerik
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Menghapus missing value
df = df.dropna()

In [4]:
df = df.drop("customerID", axis=1)

In [5]:
# Mengubah target menjadi 0 dan 1
df["Churn"] = df["Churn"].map({"No":0, "Yes":1})

# One Hot Encoding
df = pd.get_dummies(df, drop_first=True)

In [6]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [7]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [9]:
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score

# Prediksi kelas
y_pred = rf.predict(X_te)

# Prediksi probabilitas
y_prob = rf.predict_proba(X_te)[:,1]

# Classification Report
print(classification_report(y_te, y_pred))

# ROC AUC
roc = roc_auc_score(y_te, y_prob)
print("ROC-AUC :", roc)

              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1033
           1       0.63      0.49      0.55       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.71      1407
weighted avg       0.78      0.79      0.78      1407

ROC-AUC : 0.8198746188610092


In [10]:
# Probabilitas churn
prob_churn = rf.predict_proba(X_te)[:,1]

hasil = pd.DataFrame({
    "Actual": y_te.values,
    "Probabilitas_Churn": prob_churn
})

print(hasil.head(10))

   Actual  Probabilitas_Churn
0       0            0.010000
1       0            0.700000
2       0            0.016667
3       1            0.045793
4       0            0.156667
5       1            0.226667
6       0            0.020000
7       0            0.200000
8       1            0.703333
9       0            0.000000


## Kesimpulan
Model Random Forest dapat digunakan untuk memprediksi apakah pelanggan berpotensi melakukan *churn*. Hasil evaluasi menunjukkan bahwa model memiliki kemampuan yang cukup baik dalam membedakan pelanggan yang akan churn dan yang tidak. Nilai probabilitas dari `predict_proba()` juga dapat digunakan untuk mengetahui pelanggan yang memiliki risiko churn lebih tinggi. Dengan informasi tersebut, perusahaan dapat mengambil langkah lebih awal untuk mempertahankan pelanggan dan mengurangi jumlah pelanggan yang berhenti menggunakan layanan.
